In [1]:
from pathlib import Path

import pandas as pd


project_root = Path("/home/svu/e1538713/CodeNo0")
if not project_root.exists():
    project_root = Path("/nfs/home/svu/e1538713/CodeNo0")

hgnc_csv = project_root / "data" / "interim" / "hgnc_gene_symbol_indexed.csv"
parse_gene_metadata_csv = project_root / "data" / "interim" / "Parse_10M_PBMC_PBS_gene_metadata.csv"

print(f"HGNC CSV: {hgnc_csv}")
print(f"Parse gene metadata CSV: {parse_gene_metadata_csv}")

HGNC CSV: /home/svu/e1538713/CodeNo0/data/interim/hgnc_gene_symbol_indexed.csv
Parse gene metadata CSV: /home/svu/e1538713/CodeNo0/data/interim/Parse_10M_PBMC_PBS_gene_metadata.csv


In [2]:
hgnc = pd.read_csv(hgnc_csv, dtype=str, keep_default_na=False)
parse_gene_metadata = pd.read_csv(parse_gene_metadata_csv, dtype=str, keep_default_na=False)

print(f"HGNC rows: {len(hgnc):,}")
print(f"Parse gene metadata rows: {len(parse_gene_metadata):,}")
print(f"HGNC columns: {list(hgnc.columns)}")
print(f"Parse gene metadata columns: {list(parse_gene_metadata.columns)}")

HGNC rows: 19,295
Parse gene metadata rows: 40,352
HGNC columns: ['Symbol', 'Index', 'Name', 'ID', 'URL']
Parse gene metadata columns: ['Symbol', 'n_cells']


In [3]:
def find_symbol_column(df, table_name):
    if "Symbol" in df.columns:
        return "Symbol"
    if "symbol" in df.columns:
        return "symbol"
    raise ValueError(f"{table_name} 中没有找到 Symbol 或 symbol 列")


hgnc_symbol_col = find_symbol_column(hgnc, "hgnc_gene_symbol_indexed.csv")
parse_symbol_col = find_symbol_column(parse_gene_metadata, "Parse_10M_PBMC_PBS_gene_metadata.csv")

parse_symbols = set(parse_gene_metadata[parse_symbol_col])
missing_mask = ~hgnc[hgnc_symbol_col].isin(parse_symbols)
missing_hgnc = hgnc.loc[missing_mask].copy()

print(f"HGNC Symbol 数量: {len(hgnc):,}")
print(f"Parse Symbol 唯一数量: {len(parse_symbols):,}")
print(f"未找到完全一致对应基因数量: {len(missing_hgnc):,}")

if missing_hgnc.empty:
    print("所有 hgnc_gene_symbol_indexed 中的基因都可以在 Parse_10M_PBMC_PBS_gene_metadata 中找到完全一致的 Symbol。")
else:
    print("未找到对应基因的 HGNC 基因如下：")
    print(missing_hgnc.to_string(index=False))

HGNC Symbol 数量: 19,295
Parse Symbol 唯一数量: 40,352
未找到完全一致对应基因数量: 1,506
未找到对应基因的 HGNC 基因如下：
    Symbol Index                                                                                    Name         ID                                                                     URL
     ACKR5   154                                                           atypical chemokine receptor 5 HGNC:13708 https://www.genenames.org/data/gene-symbol-report/#!/hgnc_id/HGNC:13708
    ACTBL2   206                                                                       actin beta like 2 HGNC:17780 https://www.genenames.org/data/gene-symbol-report/#!/hgnc_id/HGNC:17780
    ACTL7A   212                                                                           actin like 7A   HGNC:161   https://www.genenames.org/data/gene-symbol-report/#!/hgnc_id/HGNC:161
     ACTL9   215                                                                            actin like 9 HGNC:28494 https://www.genenames.org/data/gene-symbol